# Outlier Analysis

Comprehensive multi-method outlier detection for the TiAl features dataset.

### Methods

1. Z-Score
2. Modified Z-Score (MAD-based)
3. IQR / Tukey's Fences
4. Isolation Forest
5. Local Outlier Factor (LOF)
6. Mahalanobis Distance

### Outputs

- Descriptive statistics
- Per-method outlier results
- Multi-method consensus vote matrix
- Diagnostic visualizations
- CSV report

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.spatial.distance import mahalanobis
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

## Load Dataset

In [ ]:
# DATA_PATH
DATA_PATH = "https://raw.githubusercontent.com/username/repository/main/Data/Processed/Features_dataset.csv" # Placeholder URL
REPORT_DIR = "../reports"

os.makedirs(REPORT_DIR, exist_ok=True)

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset shape: {df.shape}")
except Exception as e:
    print(f"Could not load data from URL: {e}")
    print("Please ensure the DATA_PATH points to a valid CSV file.")

## Feature Selection

In [ ]:
binary_cols = [
    c for c in df.columns
    if df[c].dropna().isin([0, 1]).all()
]

exclude = {"sample_name", "mill_type", *binary_cols}

numeric_cols = [
    c for c in df.columns
    if c not in exclude
]

df_num = df[numeric_cols].copy()

print(f"Features analysed     : {numeric_cols}")
print(f"Binary features skipped: {binary_cols}")
print(f"Samples                : {len(df)}")

## Descriptive Satistics

In [ ]:
desc = df_num.describe().T
desc["skew"] = df_num.skew()
desc["kurtosis"] = df_num.kurtosis()

desc.round(4)

In [ ]:
# Calculate Z-scores
z_scores = df_num.apply(stats.zscore)

# Identify outliers
z3_mask = z_scores.abs() > 3
z2_mask = z_scores.abs() > 2

# Moderate outliers = >2 and <=3
z2_only = z2_mask & ~z3_mask


# ─────────────────────────────────────────────
# Extreme outliers: |Z| > 3
# ─────────────────────────────────────────────

z3_results = []

for col in numeric_cols:
    mask = z3_mask[col]

    for idx in df.index[mask]:
        z3_results.append({
            "Feature": col,
            "Sample": df.loc[idx, "sample_name"],
            "Value": df_num.loc[idx, col],
            "Z_Score": z_scores.loc[idx, col],
        })

z3_results = pd.DataFrame(z3_results)

print(z3_results)


# ─────────────────────────────────────────────
# Moderate outliers: 2 < |Z| <= 3
# ─────────────────────────────────────────────

z2_results = []

for col in numeric_cols:
    mask = z2_only[col]

    for idx in df.index[mask]:
        z2_results.append({
            "Feature": col,
            "Sample": df.loc[idx, "sample_name"],
            "Value": df_num.loc[idx, col],
            "Z_Score": z_scores.loc[idx, col],
        })

z2_results = pd.DataFrame(z2_results)

print(z2_results)

In [ ]:
def modified_z_score(series):
    """Iglewicz & Hoaglin (1993) modified Z-score using MAD."""
    median = series.median()
    mad = np.median(np.abs(series - median))

    if mad == 0:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return 0.6745 * (series - median) / mad

mz_scores = df_num.apply(modified_z_score)

mz_mask = mz_scores.abs() > 3.5

mz_results = []

for col in numeric_cols:
    mask = mz_mask[col]

    for idx in df.index[mask]:
        mz_results.append({
            "Feature": col,
            "Sample": df.loc[idx, "sample_name"],
            "Value": df_num.loc[idx, col],
            "Modified_Z": mz_scores.loc[idx, col],
        })

mz_results = pd.DataFrame(mz_results)

mz_results

In [ ]:
iqr_results = {}

for col in numeric_cols:
    Q1 = df_num[col].quantile(0.25)
    Q3 = df_num[col].quantile(0.75)
    IQR = Q3 - Q1

    iqr_results[col] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_1.5": Q1 - 1.5 * IQR,
        "upper_1.5": Q3 + 1.5 * IQR,
        "lower_3.0": Q1 - 3.0 * IQR,
        "upper_3.0": Q3 + 3.0 * IQR,
    }

iqr_15_mask = pd.DataFrame(
    False,
    index=df.index,
    columns=numeric_cols
)

iqr_30_mask = pd.DataFrame(
    False,
    index=df.index,
    columns=numeric_cols
)

for col in numeric_cols:
    r = iqr_results[col]

    iqr_15_mask[col] = (
        (df_num[col] < r["lower_1.5"]) |
        (df_num[col] > r["upper_1.5"])
    )

    iqr_30_mask[col] = (
        (df_num[col] < r["lower_3.0"]) |
        (df_num[col] > r["upper_3.0"])
    )


iqr_15_mask.sum().to_frame("Outliers")

iqr_30_mask.sum().to_frame("Extreme Outliers")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)

iso_forest = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_labels = iso_forest.fit_predict(X_scaled)
iso_scores = iso_forest.decision_function(X_scaled)

iso_df = pd.DataFrame({
    "Sample": df["sample_name"].values,
    "Label": np.where(
        iso_labels == -1,
        "OUTLIER",
        "Inlier"
    ),
    "Anomaly_Score": np.round(iso_scores, 4),
}).sort_values("Anomaly_Score")

iso_df

n_iso = (iso_labels == -1).sum()

print(f"Outliers flagged: {n_iso}")

In [ ]:
n_neighbors = min(5, len(df_num) - 1)

lof = LocalOutlierFactor(
    n_neighbors=n_neighbors,
    contamination="auto"
)

lof_labels = lof.fit_predict(X_scaled)
lof_scores = -lof.negative_outlier_factor_

lof_df = pd.DataFrame({
    "Sample": df["sample_name"].values,
    "Label": np.where(
        lof_labels == -1,
        "OUTLIER",
        "Inlier"
    ),
    "LOF_Score": np.round(lof_scores, 4),
}).sort_values(
    "LOF_Score",
    ascending=False
)

lof_df

n_lof = (lof_labels == -1).sum()

print(f"Outliers flagged: {n_lof}")

In [ ]:
try:
    mean = df_num.mean().values
    cov_matrix = df_num.cov().values
    cov_inv = np.linalg.pinv(cov_matrix)

    maha_dists = np.array([
        mahalanobis(
            df_num.iloc[i].values,
            mean,
            cov_inv
        )
        for i in range(len(df_num))
    ])

    p = len(numeric_cols)

    chi2_threshold = stats.chi2.ppf(
        0.975,
        df=p
    )

    maha_outliers = (
        maha_dists >
        np.sqrt(chi2_threshold)
    )

except Exception as e:
    print(f"Mahalanobis distance failed: {e}")

    maha_outliers = np.zeros(
        len(df),
        dtype=bool
    )

    maha_dists = np.zeros(len(df))


print(f"Degrees of freedom       : {p}")
print(f"χ² threshold (97.5%)     : {chi2_threshold:.4f}")
print(f"Distance threshold       : {np.sqrt(chi2_threshold):.4f}")

maha_df = pd.DataFrame({
    "Sample": df["sample_name"].values,
    "Mahal_Distance": np.round(
        maha_dists,
        4
    ),
    "Outlier": np.where(
        maha_outliers,
        "YES",
        "no"
    ),
}).sort_values(
    "Mahal_Distance",
    ascending=False
)

maha_df

In [ ]:
vote_matrix = pd.DataFrame({
    "Sample": df["sample_name"].values,

    "Z_Score_>3":
        z3_mask.any(axis=1).astype(int),

    "Z_Score_>2":
        z2_mask.any(axis=1).astype(int),

    "Modified_Z":
        mz_mask.any(axis=1).astype(int),

    "IQR_1.5x":
        iqr_15_mask.any(axis=1).astype(int),

    "IQR_3.0x":
        iqr_30_mask.any(axis=1).astype(int),

    "IsoForest":
        (iso_labels == -1).astype(int),

    "LOF":
        (lof_labels == -1).astype(int),

    "Mahalanobis":
        maha_outliers.astype(int),
})

strict_cols = [
    "Z_Score_>3",
    "Modified_Z",
    "IQR_1.5x",
    "IsoForest",
    "LOF",
    "Mahalanobis",
]

vote_matrix["Votes_of_6"] = (
    vote_matrix[strict_cols]
    .sum(axis=1)
)

vote_matrix = vote_matrix.sort_values(
    "Votes_of_6",
    ascending=False
)

vote_matrix

consensus_outliers = vote_matrix[
    vote_matrix["Votes_of_6"] >= 3
]

consensus_outliers

## Results

In [ ]:
vote_csv_path = os.path.join(
    REPORT_DIR,
    "outlier_vote_matrix.csv"
)

vote_matrix.to_csv(
    vote_csv_path,
    index=False
)

print(f"Saved: {vote_csv_path}")

In [ ]:
# Box Plot:

n_cols = len(numeric_cols)

fig, axes = plt.subplots(
    3,
    3,
    figsize=(14, 11)
)

axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        ax = axes[i]

        bp = ax.boxplot(
            df_num[col].values,
            vert=True,
            patch_artist=True,
            widths=0.5
        )

        bp["boxes"][0].set_facecolor("#3B82F6")
        bp["boxes"][0].set_alpha(0.6)

        bp["medians"][0].set_color("#DC2626")
        bp["medians"][0].set_linewidth(2)

        for flier in bp["fliers"]:
            flier.set(
                marker="D",
                color="#DC2626",
                markersize=6
            )

        ax.set_title(
            col,
            fontweight="bold",
            fontsize=10
        )

        ax.set_xticks([])

for j in range(
    len(numeric_cols),
    len(axes)
):
    axes[j].set_visible(False)

fig.suptitle(
    "Box Plots — Outlier Detection per Feature",
    fontweight="bold",
    fontsize=13
)

fig.tight_layout()

path1 = os.path.join(
    REPORT_DIR,
    "outlier_boxplots.png"
)

fig.savefig(
    path1,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Z_score Heatmap:

fig, ax = plt.subplots(
    figsize=(14, 7)
)

z_display = z_scores.copy()

z_display.index = [
    f"{n} (Row {i})"
    for i, n in enumerate(
        df["sample_name"]
    )
]

sns.heatmap(
    z_display.T,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    annot=True,
    fmt=".1f",
    cbar_kws={"label": "Z-Score"},
    ax=ax,
    annot_kws={"size": 7}
)

ax.set_title(
    "Z-Score Heatmap — All Samples × Features",
    fontweight="bold"
)

ax.set_ylabel("Feature")
ax.set_xlabel("Sample")

plt.xticks(
    rotation=45,
    ha="right",
    fontsize=8
)

plt.yticks(fontsize=9)

fig.tight_layout()

path2 = os.path.join(
    REPORT_DIR,
    "outlier_zscore_heatmap.png"
)

fig.savefig(
    path2,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Vote Matrix Heatmap:

fig, ax = plt.subplots(
    figsize=(12, 7)
)

vote_plot = (
    vote_matrix
    .set_index("Sample")[strict_cols]
)

sns.heatmap(
    vote_plot,
    cmap="YlOrRd",
    linewidths=0.8,
    annot=True,
    fmt="d",
    cbar_kws={
        "label": "Flagged (1=Yes)"
    },
    ax=ax,
    vmin=0,
    vmax=1
)

ax.set_title(
    "Outlier Vote Matrix — Multi-Method Consensus",
    fontweight="bold"
)

ax.set_ylabel("Sample")
ax.set_xlabel("Detection Method")

plt.yticks(
    rotation=0,
    fontsize=9
)

fig.tight_layout()

path3 = os.path.join(
    REPORT_DIR,
    "outlier_vote_heatmap.png"
)

fig.savefig(
    path3,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Isolation forest:

fig, ax = plt.subplots(
    figsize=(10, 5)
)

colors = [
    "#DC2626" if l == -1 else "#3B82F6"
    for l in iso_labels
]

sample_order = np.argsort(iso_scores)

ax.barh(
    [
        df["sample_name"].iloc[i]
        for i in sample_order
    ],
    iso_scores[sample_order],
    color=[
        colors[i]
        for i in sample_order
    ],
    edgecolor="white",
    linewidth=0.5
)

ax.axvline(
    0,
    color="grey",
    linewidth=0.8,
    linestyle="--"
)

ax.set_xlabel(
    "Anomaly Score (lower = more anomalous)",
    fontweight="bold"
)

ax.set_title(
    "Isolation Forest — Anomaly Scores",
    fontweight="bold"
)

fig.tight_layout()

path4 = os.path.join(
    REPORT_DIR,
    "outlier_isolation_forest.png"
)

fig.savefig(
    path4,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Mahalanobis Distance:

fig, ax = plt.subplots(
    figsize=(10, 5)
)

colors = [
    "#DC2626" if o else "#3B82F6"
    for o in maha_outliers
]

sample_order = np.argsort(maha_dists)[::-1]

ax.barh(
    [
        df["sample_name"].iloc[i]
        for i in sample_order
    ],
    maha_dists[sample_order],
    color=[
        colors[i]
        for i in sample_order
    ],
    edgecolor="white",
    linewidth=0.5
)

if maha_threshold > 0:
    ax.axvline(
        maha_threshold,
        color="#DC2626",
        linewidth=1.5,
        linestyle="--",
        label=f"Chi-square cutoff ({maha_threshold:.2f})"
    )
    ax.legend(frameon=True, fancybox=True)

ax.set_xlabel(
    "Mahalanobis Distance",
    fontweight="bold"
)

ax.set_title(
    "Mahalanobis Distance — Multivariate Anomaly Detection",
    fontweight="bold"
)

fig.tight_layout()

path5 = os.path.join(
    REPORT_DIR,
    "outlier_mahalanobis.png"
)

fig.savefig(
    path5,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


In [ ]:
# LOF:

fig, ax = plt.subplots(
    figsize=(10, 5)
)

lof_order = np.argsort(
    lof_scores
)[::-1]

lof_colors = [
    "#DC2626"
    if lof_labels[i] == -1
    else "#3B82F6"
    for i in lof_order
]

ax.barh(
    [
        df["sample_name"].iloc[i]
        for i in lof_order
    ],
    lof_scores[lof_order],
    color=lof_colors,
    edgecolor="white",
    linewidth=0.5
)

ax.axvline(
    1.0,
    color="#DC2626",
    linewidth=1.5,
    linestyle="--",
    label="LOF = 1.0 threshold"
)

ax.set_xlabel(
    "Local Outlier Factor (higher = more anomalous)",
    fontweight="bold"
)

ax.set_title(
    "Local Outlier Factor (LOF) Scores",
    fontweight="bold"
)

ax.legend(
    frameon=True,
    fancybox=True
)

fig.tight_layout()

path6 = os.path.join(
    REPORT_DIR,
    "outlier_lof_scores.png"
)

fig.savefig(
    path6,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Summary:

summary = {
    "Z > 3": z3_mask.any(axis=1).sum(),
    "Z > 2": z2_mask.any(axis=1).sum(),
    "Modified Z": mz_mask.any(axis=1).sum(),
    "IQR 1.5×": iqr_15_mask.any(axis=1).sum(),
    "IQR 3.0×": iqr_30_mask.any(axis=1).sum(),
    "Isolation Forest": (iso_labels == -1).sum(),
    "LOF": (lof_labels == -1).sum(),
    "Mahalanobis": maha_outliers.sum(),
    "Consensus ≥3/6": len(consensus_outliers),
}

pd.Series(summary, name="Outlier Count")

consensus_outliers[
    ["Sample", "Votes_of_6"]
    + strict_cols
]